
# 04 — WGI Governance Compilation  
## Contextual / Sensitivity Only

This notebook processes the World Bank Worldwide Governance Indicators (WGI) raw package.

## Final methodological decision

WGI/governance is **not** included in the final POSet ordering variables.

The final POSet ordering variables remain the five economic/structural variables:

1. debt capacity
2. employment strength
3. R&D intensity
4. human capital
5. energy security

WGI is retained only for:

- contextual institutional interpretation;
- descriptive comparison;
- optional sensitivity checks;
- report discussion.

## What this notebook creates

The notebook extracts the six official WGI dimensions:

- Voice and Accountability
- Political Stability and Absence of Violence/Terrorism
- Government Effectiveness
- Regulatory Quality
- Rule of Law
- Control of Corruption

It then creates derived governance context scores, clearly labelled as **derived**, not official WGI indicators.

## Important report wording

Use wording like:

> WGI governance data were retained as contextual institutional information and possible sensitivity material. They were not included as final POSet ordering variables.

Do **not** write that WGI is one of the five final ordering variables.


In [1]:

# ------------------------------------------------------
# IMPORTS AND PATHS
# ------------------------------------------------------

from pathlib import Path
from datetime import datetime
import zipfile
import warnings
import re
import json

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()

RAW_FILES_DIR = PROJECT_ROOT / "Data" / "Raw" / "Raw_Files"
WGI_EXTRACT_DIR = RAW_FILES_DIR / "WGI_Extracted"

PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed" / "WGI_Governance_ContextOnly"
DIAGNOSTICS_DIR = PROJECT_ROOT / "Data" / "Diagnostics" / "04_WGI_Governance_Compilation_ContextOnly"
TABLES_DIR = PROJECT_ROOT / "Outputs" / "Tables" / "WGI_Governance_ContextOnly"
AUDIT_DIR = PROJECT_ROOT / "Data" / "Audit"

for folder in [WGI_EXTRACT_DIR, PROCESSED_DIR, DIAGNOSTICS_DIR, TABLES_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

START_YEAR = 2002
END_YEAR = 2024

print("Run ID:", RUN_ID)
print("Project root:", PROJECT_ROOT.resolve())
print("Raw files folder:", RAW_FILES_DIR.resolve())
print("WGI extract folder:", WGI_EXTRACT_DIR.resolve())
print("Processed folder:", PROCESSED_DIR.resolve())


Run ID: 20260624_183200
Project root: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24
Raw files folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files
WGI extract folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_Extracted
Processed folder: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Processed\WGI_Governance_ContextOnly


In [2]:

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

WGI_ZIP_CANDIDATES = [
    RAW_FILES_DIR / "WGI_CSV.zip",
    PROJECT_ROOT / "Data" / "Raw" / "WGI_CSV.zip",
    PROJECT_ROOT / "WGI_CSV.zip",
]

# These are the World Bank WGI indicator codes for the estimate values.
# We keep estimate series only, not percentile/rank/std-error variants.
WGI_SERIES_CODE_TO_COLUMN = {
    "VA.EST": "wgi_va_score",
    "PV.EST": "wgi_pv_score",
    "GE.EST": "wgi_ge_score",
    "RQ.EST": "wgi_rq_score",
    "RL.EST": "wgi_rl_score",
    "CC.EST": "wgi_cc_score",
}

WGI_COLUMN_LABELS = {
    "wgi_va_score": "Voice and Accountability",
    "wgi_pv_score": "Political Stability and Absence of Violence/Terrorism",
    "wgi_ge_score": "Government Effectiveness",
    "wgi_rq_score": "Regulatory Quality",
    "wgi_rl_score": "Rule of Law",
    "wgi_cc_score": "Control of Corruption",
}

WGI_SCORE_COLUMNS = list(WGI_COLUMN_LABELS.keys())

print("WGI dimensions to extract:")
display(pd.DataFrame([
    {"series_code": k, "column": v, "label": WGI_COLUMN_LABELS[v]}
    for k, v in WGI_SERIES_CODE_TO_COLUMN.items()
]))


WGI dimensions to extract:


,series_code,column,label
0,VA.EST,wgi_va_score,Voice and Accountability
1,PV.EST,wgi_pv_score,Political Stability and Absence of Violence/Te...
2,GE.EST,wgi_ge_score,Government Effectiveness
3,RQ.EST,wgi_rq_score,Regulatory Quality
4,RL.EST,wgi_rl_score,Rule of Law
5,CC.EST,wgi_cc_score,Control of Corruption


In [3]:

# ------------------------------------------------------
# SAVE HELPER
# ------------------------------------------------------

table_inventory_rows = []

def save_table(table, file_name, title, description):
    processed_path = PROCESSED_DIR / file_name
    diagnostics_path = DIAGNOSTICS_DIR / file_name
    table_path = TABLES_DIR / file_name

    table.to_csv(processed_path, index=False)
    table.to_csv(diagnostics_path, index=False)
    table.to_csv(table_path, index=False)

    table_inventory_rows.append({
        "file_name": file_name,
        "title": title,
        "description": description,
        "rows": len(table),
        "columns": len(table.columns),
        "processed_path": str(processed_path),
        "diagnostics_path": str(diagnostics_path),
        "table_path": str(table_path),
        "created_at": RUN_TIMESTAMP,
    })

    print("Saved:", file_name)


def standardize_country_code(code):
    if pd.isna(code):
        return np.nan
    code = str(code).strip().upper()
    if code == "" or code in {"NAN", "NONE"}:
        return np.nan
    # WGI country code file usually already uses 3-letter codes.
    if re.fullmatch(r"[A-Z]{3}", code):
        return code
    return np.nan


In [4]:

# ------------------------------------------------------
# LOCATE AND UNPACK WGI RAW PACKAGE
# ------------------------------------------------------

wgi_zip_path = next((p for p in WGI_ZIP_CANDIDATES if p.exists()), None)

if wgi_zip_path is None:
    raise FileNotFoundError(
        "Could not find WGI_CSV.zip. Tried:\n"
        + "\n".join([f"- {p}" for p in WGI_ZIP_CANDIDATES])
        + "\nRun Notebook 00 first or place WGI_CSV.zip inside Data/Raw/Raw_Files/."
    )

print("Using WGI zip:", wgi_zip_path)

# Extract package.
with zipfile.ZipFile(wgi_zip_path, "r") as zf:
    zf.extractall(WGI_EXTRACT_DIR)

wgi_raw_package_inventory = pd.DataFrame([
    {
        "file_name": p.name,
        "relative_path": str(p.relative_to(PROJECT_ROOT)) if p.is_relative_to(PROJECT_ROOT) else str(p),
        "size_kb": round(p.stat().st_size / 1024, 2),
    }
    for p in sorted(WGI_EXTRACT_DIR.glob("**/*"))
    if p.is_file()
])

save_table(
    wgi_raw_package_inventory,
    "wgi_raw_package_inventory.csv",
    "WGI raw package inventory",
    "Files extracted from WGI_CSV.zip.",
)

display(wgi_raw_package_inventory)


Using WGI zip: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_CSV.zip
Saved: wgi_raw_package_inventory.csv


,file_name,relative_path,size_kb
0,WGICountry.csv,Data\Raw\Raw_Files\WGI_Extracted\WGICountry.csv,137.9000
1,WGICSV.csv,Data\Raw\Raw_Files\WGI_Extracted\WGICSV.csv,3332.4800
2,WGISeries.csv,Data\Raw\Raw_Files\WGI_Extracted\WGISeries.csv,103.9400


In [5]:

# ------------------------------------------------------
# DISCOVER WGI CSV FILES
# ------------------------------------------------------

csv_files = list(WGI_EXTRACT_DIR.glob("**/*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found after extracting WGI package to {WGI_EXTRACT_DIR}")

def find_wgi_file(name_contains):
    name_contains = name_contains.lower()
    matches = [p for p in csv_files if name_contains in p.name.lower()]
    return matches[0] if matches else None

wgi_csv_path = find_wgi_file("wgicsv")
wgi_country_path = find_wgi_file("wgicountry")
wgi_series_path = find_wgi_file("wgiseries")

# Fallback: the largest CSV is usually WGICSV.csv
if wgi_csv_path is None:
    wgi_csv_path = max(csv_files, key=lambda p: p.stat().st_size)

print("WGI data CSV:", wgi_csv_path)
print("WGI country CSV:", wgi_country_path)
print("WGI series CSV:", wgi_series_path)

wgi_file_discovery = pd.DataFrame([
    {"role": "main_data", "path": str(wgi_csv_path), "exists": wgi_csv_path is not None},
    {"role": "country_metadata", "path": str(wgi_country_path), "exists": wgi_country_path is not None},
    {"role": "series_metadata", "path": str(wgi_series_path), "exists": wgi_series_path is not None},
])

save_table(
    wgi_file_discovery,
    "wgi_file_discovery.csv",
    "WGI file discovery",
    "Detected WGI CSV files used by the notebook.",
)

display(wgi_file_discovery)


WGI data CSV: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_Extracted\WGICSV.csv
WGI country CSV: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_Extracted\WGICountry.csv
WGI series CSV: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Raw\Raw_Files\WGI_Extracted\WGISeries.csv
Saved: wgi_file_discovery.csv


,role,path,exists
0,main_data,D:\Milano Bicocca\Course Materials\1st Year\02...,True
1,country_metadata,D:\Milano Bicocca\Course Materials\1st Year\02...,True
2,series_metadata,D:\Milano Bicocca\Course Materials\1st Year\02...,True


In [6]:

# ------------------------------------------------------
# LOAD WGI DATA
# ------------------------------------------------------

wgi_raw = pd.read_csv(wgi_csv_path)

# Normalize column names by stripping spaces.
wgi_raw.columns = [str(c).strip() for c in wgi_raw.columns]

print("WGI raw shape:", wgi_raw.shape)
print("Columns:")
print(list(wgi_raw.columns)[:80])

display(wgi_raw.head())


WGI raw shape: (7776, 30)
Columns:
['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1996', '1998', '2000', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


,Country Name,Country Code,Indicator Name,Indicator Code,1996,1998,2000,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,Anguilla,AIA,Control of Corruption - Governance estimate (a...,GOV_WGI_CC.EST,NaN,NaN,NaN,NaN,NaN,0.7015,1.3271,1.3353,1.3294,1.3310,1.3309,1.3319,1.3295,1.3246,1.3250,1.1646,1.1685,1.1717,1.1694,1.1732,1.1785,0.5558,0.5532,1.1763,1.1733,1.1762
1,Anguilla,AIA,Control of Corruption - Governance score (0-100),GOV_WGI_CC.SC,NaN,NaN,NaN,NaN,NaN,62.0550,74.7178,74.8834,74.7648,74.7963,74.7952,74.8142,74.7665,74.6674,74.6750,71.4287,71.5074,71.5709,71.5255,71.6027,71.7090,59.1041,59.0521,71.6659,71.6045,71.6636
2,Anguilla,AIA,Control of Corruption - Lower bound of the 90%...,GOV_WGI_CC.SC_LB,NaN,NaN,NaN,NaN,NaN,47.6063,60.2690,60.4347,60.3160,60.3476,60.3465,60.3655,60.3178,60.2187,60.2262,56.9800,57.0587,57.1222,57.0768,57.1540,57.2603,44.6554,44.6034,57.2171,57.1557,57.2149
3,Anguilla,AIA,Control of Corruption - Number of sources,GOV_WGI_CC.SR,NaN,NaN,NaN,NaN,NaN,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
4,Anguilla,AIA,Control of Corruption - Standard error of the ...,GOV_WGI_CC.SE,NaN,NaN,NaN,NaN,NaN,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353,0.4353


In [7]:

# ------------------------------------------------------
# NORMALIZE WGI STRUCTURE
# ------------------------------------------------------
# World Bank WGI bulk CSV can appear in slightly different wide formats.
# This cell handles the common DataBank format:
# Country Name | Country Code | Series Name | Series Code | 1996 [YR1996] | ...

column_map = {}

for col in wgi_raw.columns:
    c = col.strip().lower()
    if c in {"country name", "country"}:
        column_map[col] = "country_name"
    elif c in {"country code", "countrycode"}:
        column_map[col] = "Country"
    elif c in {"series name", "indicator name"}:
        column_map[col] = "series_name"
    elif c in {"series code", "indicator code"}:
        column_map[col] = "series_code"

wgi_norm = wgi_raw.rename(columns=column_map).copy()

required = {"Country", "series_code"}
missing = required - set(wgi_norm.columns)

if missing:
    raise ValueError(
        f"WGI main file missing expected columns after normalization: {missing}\n"
        f"Available columns: {list(wgi_norm.columns)}"
    )

# Identify year columns like "2002 [YR2002]" or plain "2002".
year_cols = []
for col in wgi_norm.columns:
    match = re.search(r"(19\d{2}|20\d{2})", str(col))
    if match:
        year = int(match.group(1))
        if START_YEAR <= year <= END_YEAR:
            year_cols.append(col)

if not year_cols:
    raise ValueError("No WGI year columns found in the requested year range.")

id_cols = [c for c in ["country_name", "Country", "series_name", "series_code"] if c in wgi_norm.columns]

wgi_long = wgi_norm[id_cols + year_cols].melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year_column",
    value_name="Value",
)

wgi_long["Year"] = wgi_long["year_column"].astype(str).str.extract(r"(19\d{2}|20\d{2})").astype(int)
wgi_long["Country"] = wgi_long["Country"].apply(standardize_country_code)

# WGI uses ".." or empty strings for missing values.
wgi_long["Value"] = (
    wgi_long["Value"]
    .replace({"..": np.nan, "": np.nan, " ": np.nan})
)
wgi_long["Value"] = pd.to_numeric(wgi_long["Value"], errors="coerce")

wgi_long = (
    wgi_long
    .dropna(subset=["Country", "Year", "series_code", "Value"])
    .query("Year >= @START_YEAR and Year <= @END_YEAR")
    .copy()
)

# Keep only six official WGI estimate dimensions.
wgi_long = wgi_long[wgi_long["series_code"].isin(WGI_SERIES_CODE_TO_COLUMN.keys())].copy()

wgi_long["wgi_dimension_column"] = wgi_long["series_code"].map(WGI_SERIES_CODE_TO_COLUMN)
wgi_long["wgi_dimension_label"] = wgi_long["wgi_dimension_column"].map(WGI_COLUMN_LABELS)

print("WGI long rows:", len(wgi_long))
print("Countries:", wgi_long["Country"].nunique())
print("Years:", wgi_long["Year"].min(), "-", wgi_long["Year"].max())
print("Series:", sorted(wgi_long["series_code"].unique()))

save_table(
    wgi_long,
    "wgi_official_six_dimensions_long.csv",
    "WGI official six dimensions long format",
    "Long-format WGI official estimate values for six governance dimensions.",
)

display(wgi_long.head())


WGI long rows: 0
Countries: 0
Years: nan - nan
Series: []
Saved: wgi_official_six_dimensions_long.csv


,country_name,Country,series_name,series_code,year_column,Value,Year,wgi_dimension_column,wgi_dimension_label


In [8]:

# ------------------------------------------------------
# WIDE COUNTRY-YEAR WGI SCORES
# ------------------------------------------------------

wgi_country_year_scores_wide = (
    wgi_long
    .pivot_table(
        index=["Country", "Year"],
        columns="wgi_dimension_column",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)

# Ensure all six columns exist.
for col in WGI_SCORE_COLUMNS:
    if col not in wgi_country_year_scores_wide.columns:
        wgi_country_year_scores_wide[col] = np.nan

wgi_country_year_scores_wide = wgi_country_year_scores_wide[
    ["Country", "Year"] + WGI_SCORE_COLUMNS
].sort_values(["Country", "Year"]).reset_index(drop=True)

save_table(
    wgi_country_year_scores_wide,
    "wgi_country_year_scores_wide.csv",
    "WGI country-year scores wide",
    "Country-year WGI scores for the six official governance dimensions.",
)

display(wgi_country_year_scores_wide.head())


Saved: wgi_country_year_scores_wide.csv


wgi_dimension_column,Country,Year,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score


In [9]:

# ------------------------------------------------------
# COVERAGE DIAGNOSTICS
# ------------------------------------------------------

wgi_coverage_by_country = (
    wgi_country_year_scores_wide
    .assign(available_dimensions=lambda d: d[WGI_SCORE_COLUMNS].notna().sum(axis=1))
    .groupby("Country")
    .agg(
        min_year=("Year", "min"),
        max_year=("Year", "max"),
        country_year_rows=("Year", "nunique"),
        mean_available_dimensions=("available_dimensions", "mean"),
        full_six_dimension_years=("available_dimensions", lambda x: int((x == 6).sum())),
    )
    .reset_index()
    .sort_values(["full_six_dimension_years", "Country"], ascending=[False, True])
)

wgi_coverage_by_year = (
    wgi_country_year_scores_wide
    .assign(available_dimensions=lambda d: d[WGI_SCORE_COLUMNS].notna().sum(axis=1))
    .groupby("Year")
    .agg(
        countries=("Country", "nunique"),
        complete_six_dimension_countries=("available_dimensions", lambda x: int((x == 6).sum())),
        mean_available_dimensions=("available_dimensions", "mean"),
    )
    .reset_index()
    .sort_values("Year")
)

coverage_dim_rows = []
for col in WGI_SCORE_COLUMNS:
    temp = wgi_country_year_scores_wide[["Country", "Year", col]].copy()
    coverage_dim_rows.append({
        "dimension": col,
        "label": WGI_COLUMN_LABELS[col],
        "non_missing_rows": int(temp[col].notna().sum()),
        "countries": int(temp.loc[temp[col].notna(), "Country"].nunique()),
        "min_year": int(temp.loc[temp[col].notna(), "Year"].min()) if temp[col].notna().any() else np.nan,
        "max_year": int(temp.loc[temp[col].notna(), "Year"].max()) if temp[col].notna().any() else np.nan,
    })

wgi_coverage_by_dimension = pd.DataFrame(coverage_dim_rows)

wgi_coverage_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "run_timestamp": RUN_TIMESTAMP,
    "rows_country_year": len(wgi_country_year_scores_wide),
    "countries": wgi_country_year_scores_wide["Country"].nunique(),
    "min_year": wgi_country_year_scores_wide["Year"].min(),
    "max_year": wgi_country_year_scores_wide["Year"].max(),
    "dimensions": len(WGI_SCORE_COLUMNS),
    "wgi_role": "contextual_or_sensitivity_only_not_final_ordering",
}])

save_table(
    wgi_coverage_by_country,
    "wgi_coverage_by_country.csv",
    "WGI coverage by country",
    "Country-level WGI six-dimension coverage diagnostics.",
)

save_table(
    wgi_coverage_by_year,
    "wgi_coverage_by_year.csv",
    "WGI coverage by year",
    "Year-level WGI six-dimension coverage diagnostics.",
)

save_table(
    wgi_coverage_by_dimension,
    "wgi_coverage_by_dimension.csv",
    "WGI coverage by dimension",
    "Dimension-level WGI coverage diagnostics.",
)

save_table(
    wgi_coverage_summary,
    "wgi_coverage_summary.csv",
    "WGI coverage summary",
    "Top-level WGI coverage summary.",
)

display(wgi_coverage_summary)
display(wgi_coverage_by_dimension)
display(wgi_coverage_by_year.head())


Saved: wgi_coverage_by_country.csv
Saved: wgi_coverage_by_year.csv
Saved: wgi_coverage_by_dimension.csv
Saved: wgi_coverage_summary.csv


,run_id,run_timestamp,rows_country_year,countries,min_year,max_year,dimensions,wgi_role
0,20260624_183200,2026-06-24 18:32:00,0,0,NaN,NaN,6,contextual_or_sensitivity_only_not_final_ordering


,dimension,label,non_missing_rows,countries,min_year,max_year
0,wgi_va_score,Voice and Accountability,0,0,NaN,NaN
1,wgi_pv_score,Political Stability and Absence of Violence/Te...,0,0,NaN,NaN
2,wgi_ge_score,Government Effectiveness,0,0,NaN,NaN
3,wgi_rq_score,Regulatory Quality,0,0,NaN,NaN
4,wgi_rl_score,Rule of Law,0,0,NaN,NaN
5,wgi_cc_score,Control of Corruption,0,0,NaN,NaN


,Year,countries,complete_six_dimension_countries,mean_available_dimensions


In [10]:

# ------------------------------------------------------
# CORRELATION MATRIX
# ------------------------------------------------------

wgi_six_dimension_correlation_matrix = (
    wgi_country_year_scores_wide[WGI_SCORE_COLUMNS]
    .corr()
    .reset_index()
    .rename(columns={"index": "dimension"})
)

save_table(
    wgi_six_dimension_correlation_matrix,
    "wgi_six_dimension_correlation_matrix.csv",
    "WGI six-dimension correlation matrix",
    "Correlation matrix for the six official WGI estimate dimensions.",
)

display(wgi_six_dimension_correlation_matrix)


Saved: wgi_six_dimension_correlation_matrix.csv


wgi_dimension_column,wgi_dimension_column,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score
0,wgi_va_score,NaN,NaN,NaN,NaN,NaN,NaN
1,wgi_pv_score,NaN,NaN,NaN,NaN,NaN,NaN
2,wgi_ge_score,NaN,NaN,NaN,NaN,NaN,NaN
3,wgi_rq_score,NaN,NaN,NaN,NaN,NaN,NaN
4,wgi_rl_score,NaN,NaN,NaN,NaN,NaN,NaN
5,wgi_cc_score,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:

# ------------------------------------------------------
# DERIVED GOVERNANCE CONTEXT SCORES
# ------------------------------------------------------
# These are NOT official WGI indicators.
# They are derived contextual/sensitivity scores.

wgi_context = wgi_country_year_scores_wide.copy()

# Simple mean composite of the six dimensions.
wgi_context["wgi_six_dimension_mean_score"] = wgi_context[WGI_SCORE_COLUMNS].mean(axis=1, skipna=True)
wgi_context["wgi_six_dimension_available_count"] = wgi_context[WGI_SCORE_COLUMNS].notna().sum(axis=1)

# Rescale the simple mean to 0–100 within all available country-year rows.
# Higher = better. This is contextual, not a final POSet ordering variable.
mean_min = wgi_context["wgi_six_dimension_mean_score"].min()
mean_max = wgi_context["wgi_six_dimension_mean_score"].max()

if pd.isna(mean_min) or pd.isna(mean_max) or mean_max == mean_min:
    wgi_context["governance_context_score_0_100"] = np.nan
else:
    wgi_context["governance_context_score_0_100"] = (
        (wgi_context["wgi_six_dimension_mean_score"] - mean_min)
        / (mean_max - mean_min)
        * 100
    )

wgi_context["governance_context_role"] = "contextual_or_sensitivity_only_not_final_ordering"

save_table(
    wgi_context,
    "wgi_governance_context_country_year.csv",
    "WGI governance context country-year dataset",
    "Country-year WGI six dimensions and derived contextual governance scores. Not a final ordering variable.",
)

# For compatibility with earlier notebooks, also save a file with a familiar name.
compat = wgi_context.rename(columns={
    "governance_context_score_0_100": "governance_capacity_score_0_100"
}).copy()
compat["governance_capacity_score_0_100_role"] = "contextual_or_sensitivity_only_not_final_ordering"

save_table(
    compat,
    "wgi_final_project_ready_country_year_2002_2024.csv",
    "WGI project-ready context dataset",
    "Compatibility output containing governance_capacity_score_0_100 as context/sensitivity only.",
)

display(wgi_context.head())


Saved: wgi_governance_context_country_year.csv
Saved: wgi_final_project_ready_country_year_2002_2024.csv


wgi_dimension_column,Country,Year,wgi_va_score,wgi_pv_score,wgi_ge_score,wgi_rq_score,wgi_rl_score,wgi_cc_score,wgi_six_dimension_mean_score,wgi_six_dimension_available_count,governance_context_score_0_100,governance_context_role


In [12]:

# ------------------------------------------------------
# REPORT-READY METHODOLOGICAL NOTE
# ------------------------------------------------------

wgi_methodological_note = pd.DataFrame([
    {
        "topic": "WGI role",
        "report_text": (
            "Worldwide Governance Indicators were retained as contextual institutional information "
            "and optional sensitivity material. They were not included among the five final POSet "
            "ordering variables."
        ),
    },
    {
        "topic": "Official versus derived measures",
        "report_text": (
            "The six WGI dimensions are official World Bank governance indicators. Any combined "
            "governance score used in this project is a derived contextual composite and should not "
            "be described as an official WGI index."
        ),
    },
    {
        "topic": "Reason for exclusion from final ordering",
        "report_text": (
            "Governance was excluded from the final POSet ordering to keep the main structural "
            "framework focused on five economic and socio-structural capacity variables and to avoid "
            "mixing institutional context directly into the final ordering rule."
        ),
    },
])

save_table(
    wgi_methodological_note,
    "wgi_methodological_note.csv",
    "WGI methodological note",
    "Report-ready wording for the WGI contextual/sensitivity role.",
)

display(wgi_methodological_note)


Saved: wgi_methodological_note.csv


,topic,report_text
0,WGI role,Worldwide Governance Indicators were retained ...
1,Official versus derived measures,The six WGI dimensions are official World Bank...
2,Reason for exclusion from final ordering,Governance was excluded from the final POSet o...


In [13]:

# ------------------------------------------------------
# AUDIT WORKBOOK
# ------------------------------------------------------

audit_xlsx = AUDIT_DIR / "04_WGI_Governance_Compilation_ContextOnly_Audit.xlsx"

with pd.ExcelWriter(audit_xlsx, engine="xlsxwriter") as writer:
    wgi_raw_package_inventory.to_excel(writer, sheet_name="raw_inventory", index=False)
    wgi_file_discovery.to_excel(writer, sheet_name="file_discovery", index=False)
    wgi_coverage_summary.to_excel(writer, sheet_name="coverage_summary", index=False)
    wgi_coverage_by_dimension.to_excel(writer, sheet_name="coverage_dimension", index=False)
    wgi_coverage_by_year.to_excel(writer, sheet_name="coverage_year", index=False)
    wgi_coverage_by_country.to_excel(writer, sheet_name="coverage_country", index=False)
    wgi_six_dimension_correlation_matrix.to_excel(writer, sheet_name="correlation_matrix", index=False)
    wgi_context.head(5000).to_excel(writer, sheet_name="context_preview", index=False)
    wgi_methodological_note.to_excel(writer, sheet_name="method_note", index=False)
    pd.DataFrame(table_inventory_rows).to_excel(writer, sheet_name="table_inventory", index=False)

print("Audit workbook saved:", audit_xlsx.resolve())


Audit workbook saved: D:\Milano Bicocca\Course Materials\1st Year\02 - Data Science Lab - 2526-1-FDS02Q003\03 Projects\Index\Notebooks\Run_24\Data\Audit\04_WGI_Governance_Compilation_ContextOnly_Audit.xlsx


In [14]:

# ------------------------------------------------------
# FINAL COMPLETION CHECK
# ------------------------------------------------------

table_inventory = pd.DataFrame(table_inventory_rows)
table_inventory.to_csv(PROCESSED_DIR / "wgi_context_table_inventory.csv", index=False)
table_inventory.to_csv(DIAGNOSTICS_DIR / "wgi_context_table_inventory.csv", index=False)

expected_outputs = [
    "wgi_official_six_dimensions_long.csv",
    "wgi_country_year_scores_wide.csv",
    "wgi_governance_context_country_year.csv",
    "wgi_final_project_ready_country_year_2002_2024.csv",
    "wgi_coverage_summary.csv",
    "wgi_coverage_by_country.csv",
    "wgi_coverage_by_year.csv",
    "wgi_coverage_by_dimension.csv",
    "wgi_six_dimension_correlation_matrix.csv",
    "wgi_methodological_note.csv",
]

output_check = pd.DataFrame([
    {
        "file_name": f,
        "processed_exists": (PROCESSED_DIR / f).exists(),
        "diagnostics_exists": (DIAGNOSTICS_DIR / f).exists(),
        "tables_exists": (TABLES_DIR / f).exists(),
    }
    for f in expected_outputs
])

output_check.to_csv(DIAGNOSTICS_DIR / "output_check.csv", index=False)

print("04 WGI GOVERNANCE COMPILATION — CONTEXT ONLY COMPLETE")
print("=" * 90)
display(output_check)

print("\nKey outputs to inspect/send:")
print("- wgi_coverage_summary.csv")
print("- wgi_six_dimension_correlation_matrix.csv")
print("- wgi_methodological_note.csv")
print("- 04_WGI_Governance_Compilation_ContextOnly_Audit.xlsx")

print("\nNext notebook:")
print("05_Volatility_Features_2002.ipynb")


04 WGI GOVERNANCE COMPILATION — CONTEXT ONLY COMPLETE


,file_name,processed_exists,diagnostics_exists,tables_exists
0,wgi_official_six_dimensions_long.csv,True,True,True
1,wgi_country_year_scores_wide.csv,True,True,True
2,wgi_governance_context_country_year.csv,True,True,True
3,wgi_final_project_ready_country_year_2002_2024...,True,True,True
4,wgi_coverage_summary.csv,True,True,True
5,wgi_coverage_by_country.csv,True,True,True
6,wgi_coverage_by_year.csv,True,True,True
7,wgi_coverage_by_dimension.csv,True,True,True
8,wgi_six_dimension_correlation_matrix.csv,True,True,True
9,wgi_methodological_note.csv,True,True,True



Key outputs to inspect/send:
- wgi_coverage_summary.csv
- wgi_six_dimension_correlation_matrix.csv
- wgi_methodological_note.csv
- 04_WGI_Governance_Compilation_ContextOnly_Audit.xlsx

Next notebook:
05_Volatility_Features_2002.ipynb
